In [0]:
_checkpoints = "dbfs:/Volumes/workspace/retails/raw_data/_checkpoints/retails/dev/gold/orders"

In [0]:
# -- # Read clean orders from silver
silver_orders_df = (
    spark.readStream \
        .format("delta") \
        .option("readChangeFeed", "true") \
        .table("retails.silver.orders_cleaned")
        .select("order_id", "order_customer_id", "order_date", "order_status")
)

# silver_orders_df = silver_orders_df.withWatermark("ingestion_ts", "1 day")
# display(silver_orders_df.count())

In [0]:
# -- # Read clean order items from silver
fact_order_items_df = (
    spark.read \
        .format("delta") \
        # .option("readChangeFeed", "true") \
        .table("retails.gold.fact_order_items") \
        .select("order_item_id", "order_id", "quantity", "subtotal")
)

# fact_order_items_df = fact_order_items_df.withWatermark("created_ts", "1 day")

# display(silver_order_items_df.count())

In [0]:
dim_dates_df = (
    spark.read \
        .format("delta") \
        .table("retails.gold.dim_dates")
        .select("date_key","full_date")
)

# display(dim_dates_df.count())

In [0]:
dim_customers_df = (
    spark.read \
        .format("delta") \
        .table("retails.gold.dim_customers")
        .select("customer_key", "customer_id")
)

# display(dim_customers_df.count())

In [0]:
from pyspark.sql import functions as F

gold_orders_df = (
    silver_orders_df.alias("oc")
    .join(
        dim_dates_df.alias("dd"),
        F.col("oc.order_date") == F.col("dd.full_date"),
        how = "inner"
    )
    .join(
        dim_customers_df.alias("dc"),
        F.col("oc.order_customer_id") == F.col("dc.customer_id"),
        how = "inner"
    )
    .join(
        fact_order_items_df.alias("oic"),
        F.col("oc.order_id") == F.col("oic.order_id"),
        how = "left"
    )
    .groupBy(
        "oc.order_id",
        "dd.date_key",
        "dc.customer_key",
        "oc.order_status"
    )
    .agg(
        F.expr("coalesce(sum(quantity), 0)").alias("total_items"),
        F.expr("coalesce(sum(subtotal), 0)").alias("total_order_amount")
    )
    .select(
        "oc.order_id",
        F.col("dd.date_key").alias("order_date_key"),
        "dc.customer_key",
        "oc.order_status",
        "total_items",
        F.col("total_order_amount").cast("decimal(10,2)")
    )
)

In [0]:
gold_orders_df = gold_orders_df.withColumn("created_ts", F.current_timestamp())
print(gold_orders_df.printSchema)

In [0]:
def upsert_fact_orders(batch_df, batch_id):

    batch_df.write \
        .mode("append") \
        .saveAsTable("retails.gold.fact_orders")



In [0]:
gold_orders_df.writeStream \
    .outputMode("update") \
    .foreachBatch(upsert_fact_orders) \
    .option("checkpointLocation", _checkpoints) \
    .trigger(availableNow=True) \
    .start() \
    .awaitTermination()

In [0]:
# dbutils.fs.ls(_checkpoints)
# dbutils.fs.rm(_checkpoints, True)

In [0]:
%sql
-- make silver cleaned table CDF enabled

-- ALTER TABLE retails.silver.orders_cleaned
-- SET TBLPROPERTIES (
--     delta.enableChangeDataFeed = true
-- )

-- DESCRIBE TABLE EXTENDED retails.silver.orders_cleaned;

In [0]:
%sql
-- create table retails.sql_practice.catalog_sales_sf1000
-- as 
-- select * from samples.tpcds_sf1000.catalog_sales;

-- alter table retails.sql_practice.catalog_sales_sf1000
-- cluster by (cs_bill_cdemo_sk);

-- alter table retails.sql_practice.catalog_sales_sf1000
-- cluster by NONE;

-- OPTIMIZE retails.sql_practice.catalog_sales_sf1000 FULL;

-- describe history retails.sql_practice.catalog_sales_sf1000;

-- select distinct cs_sold_date_sk from retails.sql_practice.catalog_sales_sf1000;

-- select * from retails.sql_practice.catalog_sales_sf1000 where cs_sold_date_sk=2450943 and cs_ship_date_sk=2450966;

-- select * from retails.sql_practice.catalog_sales_sf1000 where cs_bill_cdemo_sk=606681;

In [0]:
%sql
-- create table retails.sql_practice.tpch_orders
-- as 
-- select * from samples.tpch.orders;

-- ALTER TABLE retails.sql_practice.tpch_orders
-- CLUSTER BY (o_orderdate);

-- optimize retails.sql_practice.tpch_orders FULL;

-- describe detail retails.sql_practice.tpch_orders;

-- describe history retails.sql_practice.tpch_orders;

-- analyze table retails.sql_practice.tpch_orders compute delta statistics;

    
-- select * from retails.sql_practice.tpch_orders where o_orderdate='1996-10-30';

In [0]:
%sql
-- select count(*) from retails.gold.fact_orders;
   
   -- OPTIMIZE retails.silver.orders_cleaned_zorder ZORDER BY (order_customer_id);

-- ALTER TABLE retails.sql_practice.tpch_orders
-- CLUSTER BY (o_custkey)

-- OPTIMIZE retails.silver.orders_cleaned FULL

-- describe history retails.silver.orders_cleaned
-- describe detail retails.silver.orders_cleaned

-- analyze table retails.gold.fact_orders compute statistics for columns order_status

-- DESCRIBE EXTENDED retails.gold.fact_orders;
-- SHOW TBLPROPERTIES retails.gold.fact_orders;

-- show partitions retails.gold.fact_orders;


In [0]:
query = """
    select order_id, gdd.full_date,concat(gdc.customer_fname,' ',gdc.customer_lname) as customer_name, order_status, total_items, total_order_amount 
    from retails.gold.fact_orders gfo

    inner join retails.gold.dim_dates gdd
        on gfo.order_date_key = gdd.date_key

    inner join retails.gold.dim_customers gdc
        on gfo.customer_key = gdc.customer_key

    where gdc.customer_fname like 'Mar%'
    limit 10  
"""

spark.sql(query).display()

In [0]:
# %sql
# select oc.order_id, dd.date_key, dc.customer_key, oc.order_status, coalesce(sum(oic.order_item_quantity), 0) as total_items, coalesce(sum(oic.order_item_subtotal), 0) as total_order_amount
# from retails.silver.orders_cleaned oc

#     inner join retails.gold.dim_dates dd
#         on oc.order_date = dd.full_date

#     inner join retails.gold.dim_customers dc
#         on oc.order_customer_id = dc.customer_id
    
#     left join retails.silver.order_items_cleaned oic
#         on oc.order_id = oic.order_item_order_id

#     group by oc.order_id, dd.date_key, dc.customer_key, oc.order_status

# order by order_id limit 10;

In [0]:
# %sql
# select count(*) from (
#     select oc.order_id, dd.date_key, dc.customer_key, oc.order_status, coalesce(sum(oic.order_item_quantity), 0) as total_items, coalesce(sum(oic.order_item_subtotal), 0) as total_order_amount
# from retails.silver.orders_cleaned oc

#     inner join retails.gold.dim_dates dd
#         on oc.order_date = dd.full_date

#     inner join retails.gold.dim_customers dc
#         on oc.order_customer_id = dc.customer_id
    
#     left join retails.silver.order_items_cleaned oic
#         on oc.order_id = oic.order_item_order_id

#     group by oc.order_id, dd.date_key, dc.customer_key, oc.order_status

# -- order by order_id limit 10
# )